# Retrieval-augmented generation

### Retrieval - PDF

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader, UnstructuredMarkdownLoader
from sentence_transformers import SentenceTransformer
import uuid

c:\Users\admin\Desktop\data_scientist_learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 1. Define document loaders 

In [ ]:
pdf_loader = DirectoryLoader(
    path='github_data/my_articles/',
    loader_cls=PyMuPDFLoader,
    show_progress=True
    )

markdown_loader = DirectoryLoader(
    path='github_data/markdown_files/',
    loader_cls=UnstructuredMarkdownLoader,
    show_progress=True
    )

#### 2. Define document splitter

In [19]:

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
    separators=['\n', '\n\n', "."]
    )

#### 3. Load documents and build chunks

In [17]:
documents = list(pdf_loader.lazy_load()) + list(markdown_loader.lazy_load())
chunks = splitter.split_documents(documents=documents)

100%|██████████| 1/1 [00:00<00:00, 60.44it/s]


#### 4. Create embeddings of these chunks

In [21]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = embedding_model.encode([chunk.page_content for chunk in chunks])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1999.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### 5. Set our Database (VectorDB - chromaDB)

In [42]:
client = chromadb.Client()
collection = client.get_or_create_collection(name='my_collection')
collection.add(
    documents=[chunk.page_content for chunk in chunks],
    embeddings=embeddings,
    ids=[str(uuid.uuid4()) for _ in range(len(chunks))]
)

#### 6. Ask question to vectorDB

In [57]:
query_embedding = embedding_model.encode("Tell me about your latest LLM or RAG project?")
top_chunks = collection.query(query_embeddings=[query_embedding], n_results=5)
top_chunks

{'ids': [['bc0e9a69-87f8-4e9f-bf36-8799744b674a',
   'cad3a02a-fea7-4bd4-ad84-82b3492f1234',
   'a79e2659-1ca5-42ea-b2a1-d3df7b560577',
   '39d3cb03-fd2d-46c0-bc9f-de27231501cd',
   'e0e97661-5fff-4ec5-acfe-d9810bdff7e0']],
 'embeddings': None,
 'documents': [['I License\nThis project is open-source and available for educational and portfolio purposes.\nI Acknowledgments\n•\nLangChain Community: For excellent RAG and LLM orchestration tools\n•',
   'I License\nThis project is open-source and available for educational and portfolio purposes.\nI Acknowledgments\n•\nLangChain Community: For excellent RAG and LLM orchestration tools\n•',
   'Feel free to explore and contribute to the project! If you have any questions, open an issue on the repository. ```',
   '1. Explore the project files and structure developed during the Ujucode internship.\nContributing',
   "and Retrieval-Augmented Generation (RAG) combined with an LLM to answer HR-style or educational questions based on a\ncandidate'

#### 7. Let's make it smarter with LLM intgration
- ChatGPT API
- LangChain 
- PromptEngineering

In [63]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from typing import Union, Annotated
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model='gpt-5-mini',
    temperature=0.7
)

prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template="""
        You are an AI assistant for Vijay Takbhate's portfolio.
        Context: 
        {context}

        Question: 
        {question}

        Answer the question based on the provided context. If the answer is not available in the context, ask user to put more information in question specially skills and experience.
    """
    )

class ModelResponse(BaseModel):
    answer: Annotated[str, "The answer generated by the model based on the context and question."]
    reference_links: Annotated[Union[list[str], None], "List of reference links used to answer the question."]

model = model.with_structured_output(ModelResponse)

chain = prompt | model

In [ ]:
llm_response = chain.invoke(
    input={
        "context": "\n".join([chunk for chunk in top_chunks['documents'][0]]),
        "question": "Tell me about your latest LLM or RAG project?"
    }
)

llm_response

ModelResponse(answer='My latest LLM/RAG project (developed during a Ujucode internship) is an open-source Retrieval-Augmented Generation system that combines a vector-backed retriever with an LLM to answer HR-style or educational questions using a candidate’s resume, repositories, or published articles. Key points:\n\n- Purpose: Answer HR-style or educational queries by retrieving relevant passages from resumes/repos/articles and generating concise, context-aware responses.\n- Architecture/Tooling (high level from context): RAG + LLM orchestration (acknowledges LangChain for orchestration and RAG utilities).\n- Project scope: Includes project files and structure intended for exploration during the internship; suitable for portfolio/demonstration and educational use.\n- License & contribution: The project is open-source for educational/portfolio purposes; contributions and issues are welcome on the repository.\n\nIf you’d like a more detailed description (examples, architecture diagram,

In [69]:
print(llm_response.answer)

My latest LLM/RAG project (developed during a Ujucode internship) is an open-source Retrieval-Augmented Generation system that combines a vector-backed retriever with an LLM to answer HR-style or educational questions using a candidate’s resume, repositories, or published articles. Key points:

- Purpose: Answer HR-style or educational queries by retrieving relevant passages from resumes/repos/articles and generating concise, context-aware responses.
- Architecture/Tooling (high level from context): RAG + LLM orchestration (acknowledges LangChain for orchestration and RAG utilities).
- Project scope: Includes project files and structure intended for exploration during the internship; suitable for portfolio/demonstration and educational use.
- License & contribution: The project is open-source for educational/portfolio purposes; contributions and issues are welcome on the repository.

If you’d like a more detailed description (examples, architecture diagram, exact models, embedding/vect

#### 8. Let's see the difference between only RAG and LLM + RAG

In [79]:
print('RGA RESPONSE FOR QUESTION: Tell me about your latest LLM or RAG project?')
for i in range(len(top_chunks['documents'][0])):
    print(top_chunks['documents'][0][i])

RGA RESPONSE FOR QUESTION: Tell me about your latest LLM or RAG project?
I License
This project is open-source and available for educational and portfolio purposes.
I Acknowledgments
•
LangChain Community: For excellent RAG and LLM orchestration tools
•
I License
This project is open-source and available for educational and portfolio purposes.
I Acknowledgments
•
LangChain Community: For excellent RAG and LLM orchestration tools
•
Feel free to explore and contribute to the project! If you have any questions, open an issue on the repository. ```
1. Explore the project files and structure developed during the Ujucode internship.
Contributing
and Retrieval-Augmented Generation (RAG) combined with an LLM to answer HR-style or educational questions based on a
candidate's resume, repos or published articles.
I Highlights
•


In [80]:
print('LLM + RGA RESPONSE FOR QUESTION: Tell me about your latest LLM or RAG project?')
print(llm_response.answer)

LLM + RGA RESPONSE FOR QUESTION: Tell me about your latest LLM or RAG project?
My latest LLM/RAG project (developed during a Ujucode internship) is an open-source Retrieval-Augmented Generation system that combines a vector-backed retriever with an LLM to answer HR-style or educational questions using a candidate’s resume, repositories, or published articles. Key points:

- Purpose: Answer HR-style or educational queries by retrieving relevant passages from resumes/repos/articles and generating concise, context-aware responses.
- Architecture/Tooling (high level from context): RAG + LLM orchestration (acknowledges LangChain for orchestration and RAG utilities).
- Project scope: Includes project files and structure intended for exploration during the internship; suitable for portfolio/demonstration and educational use.
- License & contribution: The project is open-source for educational/portfolio purposes; contributions and issues are welcome on the repository.

If you’d like a more det